# Photogrammetry Pipeline — API Walkthrough

This notebook is a hands-on tutorial for the `pils` + `IPA_flight` photogrammetry
stack. It's meant to be read top to bottom — every cell has comments explaining
*what* each API call does and *why* it's needed, not just *how* to call it.

**What this notebook does**

1. Locates the metadata for one flight (`pils.loader`).
2. Builds a `Flight` object and loads its drone + camera sensor data (`pils.flight.Flight`).
3. Loads the camera calibration / pipeline parameters (`pils.sensors.camera.PhotogrammetryConfig`).
4. Defines small **auxiliary helper functions** you can reuse on their own:
   - open/plot one specific video frame or image,
   - reload the final `.parquet` result of a pipeline that already ran,
   - reload an intermediate checkpoint (`dictionary_pN.ecsv`) from a pipeline that
     stopped partway through,
   - a **quick single-frame look**: photogrammetric attitude for one frame plus
     the nearest already-computed EKF attitude, for a fast sanity check without
     running the full pipeline.
5. Runs the **full photogrammetry pipeline from scratch** (`camera_obj.run_photogrammetry`).
6. Shows how to **resume** a pipeline from a saved checkpoint instead of redoing step 5.
7. Inspects and plots the final result, tying the frame viewer from step 4 back
   into the numeric result.

**The only thing you need to edit is the input cell right below (Step 0)** —
every other cell just consumes the variables defined there.


In [ ]:
# --- pils: the data-handling / IO layer -------------------------------------
#   - loaders resolve *where* a flight's files live on disk (or in the STOUT DB)
#   - Flight is the in-memory container for one flight's raw sensor data
#   - Camera / PhotogrammetryConfig are the entry points into the photogrammetry
#     pipeline itself. The pipeline logic lives in the separate IPA_flight
#     package, but Camera.run_photogrammetry() calls into it for you — you only
#     import IPA_flight directly for the small utils used by the aux functions
#     in Step 4.
from pils.loader.path import PathLoader
from pils.loader.stout import StoutLoader
from pils.flight import Flight
from pils.sensors.camera import Camera, PhotogrammetryConfig

# --- general-purpose libraries used throughout the notebook -----------------
from pathlib import Path
import cv2                 # frame-level image access
import numpy as np
import pandas as pd        # only needed for the EKF timestamp lookup in Step 4.4
import polars as pl        # pils / IPA_flight exchange data as polars DataFrames
import matplotlib.pyplot as plt

%matplotlib qt 

The X11 connection broke (error 1). Did the X11 server die?


## 0 — Inputs

Every path the rest of the notebook needs, gathered in one place. Nothing
below this cell should need editing to run the notebook on a different
flight — just change the values here.


In [ ]:
# Root folder that contains the `campaigns/` directory. Only needed if you
# locate flights by scanning the filesystem with PathLoader (see Step 1).
DATA_ROOT = "/data/POLOCALC/"

# Flight to process — the `flight_YYYYMMDD_HHMM` folder name, unique within a campaign.
FLIGHT_NAME = "flight_20251206_1530"

# YAML file with camera intrinsics + pipeline parameters. See
# pils/config/photogrammetryConfig.yaml for a fully commented template, and
# Step 3 below for what each block feeds into.
CONFIG_PATH = "/home/fastori/Desktop/POLOCALC/ARS/pils/pils/config/photogrammetryConfig.yaml"

# CSV of surveyed geodetic targets + telescope positions for the campaign
# (Emlid/RTK export, one row per named point — see GenParamFile.GeoPlot's
# docstring in IPA_flight for the exact row-naming convention it expects).
TARGETS_CSV = "/data/POLOCALC/campaigns/202511/metadata/202511_coordinates.csv"

# Where the pipeline writes its outputs: intermediate `dictionary_pN.ecsv`
# checkpoints, diagnostic plots, and the final `attitude_reconstruction*.parquet`
# (exact filename varies -- see Step 4.2). A subfolder named after the flight
# is created inside it automatically.
OUTPUT_DIR = "/home/fastori/Desktop/photogrammetry_results"

# Camera model — only needed if the YAML config has calibration blocks for
# more than one camera. Leave as None; Step 2 prints the detected model so you
# can fill this in if PhotogrammetryConfig complains about ambiguity.
CAMERA_MODEL = "sony"   # e.g. "sony" or "alvium"

## 1 — Locate the flight

`pils.loader` doesn't touch any sensor data — it only resolves a flight name
into the folder paths (`drone_data_folder_path`, `aux_data_folder_path`,
`processed_data_folder_path`) that everything else needs. Two
interchangeable implementations:

- **`PathLoader`** scans `DATA_ROOT/campaigns/**` on disk. Use this if you
  don't have (or don't want) a STOUT database connection — the common case
  when working locally.
- **`StoutLoader`** queries the STOUT campaign-management database directly,
  and transparently falls back to the same filesystem scan if the `stout`
  package isn't importable. Prefer this on a machine with STOUT configured —
  it avoids re-scanning the whole campaign tree.

Both return the exact same `dict` shape, so everything downstream is
agnostic to which one you used.


In [ ]:
# --- Option A (default): scan the filesystem --------------------------------
loader = PathLoader(DATA_ROOT)
flight_meta = loader.load_single_flight(flight_name=FLIGHT_NAME)

# --- Option B: query STOUT instead -------------------------------------------
# loader = StoutLoader()
# flight_meta = loader.load_single_flight(flight_name=FLIGHT_NAME)

if flight_meta is None:
    raise FileNotFoundError(f"Flight '{FLIGHT_NAME}' not found under {DATA_ROOT}")

flight_meta

2026-09-10 15:35:41,529 - pils.loader.path - INFO - Loading single flight: flight_id=None, flight_name=flight_20251206_1530
2026-09-10 15:35:41,529 - pils.loader.path - INFO - Loading all flights from all campaigns...
2026-09-10 15:35:41,530 - pils.loader.path - WARNING - Could not build flight dict for calibration  /data/POLOCALC/campaigns/202412/20241215/calibration : time data 'tion ' does not match format '%Y%m%d_%H%M'
2026-09-10 15:35:41,530 - pils.loader.path - WARNING - Could not build flight dict for Store-V2 /data/POLOCALC/campaigns/202412/.Spotlight-V100/Store-V2: time data '2' does not match format '%Y%m%d_%H%M'
2026-09-10 15:35:41,531 - pils.loader.path - WARNING - Could not build flight dict for 24_12_17 /data/POLOCALC/campaigns/202412/calibration_PUC/24_12_17: time data '7' does not match format '%Y%m%d_%H%M'
2026-09-10 15:35:41,531 - pils.loader.path - WARNING - Could not build flight dict for 24_12_18 /data/POLOCALC/campaigns/202412/calibration_PUC/24_12_18: time data '

{'campaign_name': '202511',
 'flight_name': 'flight_20251206_1530',
 'flight_date': '20251206',
 'takeoff_datetime': '2025-12-06T15:30:00+00:00',
 'landing_datetime': '2025-12-06T15:30:00+00:00',
 'drone_data_folder_path': '/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/drone',
 'aux_data_folder_path': '/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux',
 'processed_data_folder_path': '/data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/proc'}

## 2 — Build the `Flight` and load its sensors

`Flight` wraps `flight_meta` (just folder paths) plus a hierarchical
container, `flight.raw_data`, that gets filled in lazily by the `add_*`
methods below — nothing is read from disk until you call them.

- **`add_drone_data()`** autodetects DJI vs. BlackSquare from the folder
  contents and loads drone telemetry (+ Litchi log, if present) into
  `flight.raw_data.drone_data`.
- **`add_camera_data(use_photogrammetry=False)`** autodetects Sony
  (`.mp4` + telemetry) vs. Alvium (image sequence + `.log`) and loads
  **two** things into `flight.raw_data.payload_data`:
    - `.camera` — a `pl.DataFrame` of per-frame timestamps/telemetry, used
      for time-synchronising the camera with other sensors (`flight.sync()`).
    - `.camera_obj` — the live `Camera` instance itself. This is what
      `.run_photogrammetry(...)` (Step 5) and the frame-access helpers
      (Step 4.1) are called on.

  (`use_photogrammetry=True` instead loads an *already computed*
  photogrammetry CSV, read-only — not what we want here, since we're about
  to run the pipeline ourselves.)


In [ ]:
flight = Flight(flight_meta)

flight.add_drone_data()
flight.add_camera_data(use_photogrammetry=False)

# The Camera instance — everything from here on operates on this object.
camera_obj = flight.raw_data.payload_data.camera_obj

print(flight.raw_data)
print(camera_obj)

2026-09-10 15:35:41,548 - pils.flight - INFO - Drone : /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/drone/20251206_153038_drone.dat
2026-09-10 15:35:41,548 - pils.drones.DJIDrone - INFO - PATH: /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/drone/20251206_153038_drone.dat
2026-09-10 15:35:43,319 - pils.drones.DJIDrone - INFO - Loaded 3642 GPS messages from DAT file
2026-09-10 15:35:43,325 - pils.drones.DJIDrone - INFO - Loaded 3349 RTK messages from DAT file
2026-09-10 15:35:43,376 - pils.drones.DJIDrone - INFO - Converting timestamps to milliseconds
2026-09-10 15:35:43,479 - pils.sensors.camera - INFO - Parsing telemetry from /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (5040.2 MB, timeout=2520s)


   ⏳ telemetry_parser running... 0s / 2520s


Process Process-3:
Traceback (most recent call last):
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/fastori/anaconda3/envs/photo/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/fastori/Desktop/POLOCALC/ARS/pils/pils/sensors/camera.py", line 533, in _worker
    imu_data = parser.normalized_imu()
               ^^^^^^^^^^^^^^^^^^^^^^^
pyo3_runtime.PanicException: assertion failed: v.len() == 3
thread '<unnamed>' panicked at /home/runner/work/telemetry-parser/telemetry-parser/src/sony/mod.rs:41:9:
assertion failed: v.len() == 3
note: run with `RUST_BACKTRACE=1` environment variable to display a backtrace
2026-09-10 15:35:44,186 - pils.sensors.camera - WARNING - Sony telemetry unavailable for /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/aux/camera/20251206_153038_video.mp4 (telemetry-parser crashed (exitcode=

=== DRONE DATA ===
Drone:
shape: (6_971, 42)
┌────────────┬──────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ tick       ┆ msg_type ┆ GPS:date  ┆ GPS:time ┆ … ┆ RTK:pos_f ┆ RTK:pos_f ┆ RTK:pos_f ┆ RTK:gps_s │
│ ---        ┆ ---      ┆ ---       ┆ ---      ┆   ┆ lg_3      ┆ lg_4      ┆ lg_5      ┆ tate      │
│ i64        ┆ i64      ┆ f64       ┆ f64      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆          ┆           ┆          ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 26096229   ┆ 2096     ┆ 0.0       ┆ 0.0      ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ 28031339   ┆ 2096     ┆ 2.0251206 ┆ 152727.0 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│            ┆          ┆ e7        ┆          ┆   ┆           ┆           ┆           ┆           │
│ 28671752   ┆ 2096     ┆ 2.0251206 ┆ 152727.0

In [ ]:
# (flight.raw_data.drone_data.drone.to_pandas())

`camera_obj`'s `repr` above reports its detected `model` (`sony` /
`alvium`) and `mode` (`video` / `image_sequence`) — use that to set
`CAMERA_MODEL` in Step 0 if `PhotogrammetryConfig` complains that your YAML
has more than one camera calibration block.


## 3 — Load the photogrammetry config

`PhotogrammetryConfig` parses a YAML file into the parameters the pipeline
needs that *can't* be derived from the flight itself: camera intrinsics and
the tuning knobs for each pipeline step (see
`pils/config/photogrammetryConfig.yaml` for a fully annotated template).
Everything else the pipeline needs — targets, telescope positions, video
code, date, ... — is parsed at run time from `TARGETS_CSV` and the `Flight`
object, not from this config.

| Config attribute                       | Consumed by (pipeline step)                              |
|-----------------------------------------|------------------------------------------------------------|
| `camera_matrix`, `distortion_coeffs`    | steps p2 (finder) and p3 (PnP / MCMC attitude)              |
| `finder`                                | step p2 — `TargetFinder.finderSequential`                   |
| `pnp` / `mcmc`                          | step p3 — `AttitudeReconstruction.run_pnp` / `.run_mcmc`     |
| `drone_correlation`                     | step p4 — `DroneData.correlate_drone_photo`                 |
| `telescope`                             | step p6 — `ConcerningTelescope`                              |
| `polarization`                          | step p7 — `PolarizationAngle`                                |
| `reference_point`, `dji_base_logged`    | fixed campaign-level ENU origin + RTK base offset            |


In [ ]:
cfg = PhotogrammetryConfig(CONFIG_PATH, camera_model=CAMERA_MODEL)

print("Camera matrix:\n", cfg.camera_matrix)
print("Distortion coeffs:", cfg.distortion_coeffs)
print("Finder params:     ", cfg.finder)
print("PnP params:        ", cfg.pnp)
print("Drone corr params: ", cfg.drone_correlation)

Camera matrix:
 [[2.54302197e+03 0.00000000e+00 1.90813916e+03]
 [0.00000000e+00 2.54107129e+03 1.07218579e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion coeffs: [ 0.03171191 -0.05128733 -0.00039322  0.00012249  0.04870586]
Finder params:      {'tg_by_hand': 'click_with_tel', 'box_height': 25, 'box_width': 25, 'color_tol': 10, 'tracking_method': 'optical_flow', 'orb_nfeatures': 1000, 'refine_threshold': 0.98}
PnP params:         {'ransac': True}
Drone corr params:  {'trigger': False, 'payload': False, 'use_fft': False}


## 4 — Auxiliary functions

Small, self-contained helpers that don't require running the pipeline. Use
these to sanity-check the raw data *before* committing to a full run, or to
reload results from a run that already happened without paying the cost of
re-running anything.


### 4.1 — Open one specific frame

`camera_obj.get_frame(n)` only works for Alvium image sequences (it's a
plain `cv2.imread` on the n-th file on disk) — for Sony video it raises,
because `camera_obj.capture` is reserved for the background streaming
thread started by `add_camera_data()` (reading from the same
`cv2.VideoCapture` from two places at once would race). The helper below
works for **both** modes: it opens a *second*, independent
`cv2.VideoCapture` for random-access seeks on video, and just delegates to
`get_frame()` for image sequences.


In [ ]:
def open_frame(camera_obj: Camera, frame_number: int, color: str = "rgb", show: bool = True) -> np.ndarray:
    """Return frame `frame_number` as a BGR array; optionally display it.

    Works for both Camera operating modes:
      - image_sequence (Alvium): direct disk read via camera_obj.get_frame().
      - video (Sony): a throwaway cv2.VideoCapture is opened just for this
        seek, independent of the capture object the background reader
        thread owns, so it doesn't interfere with streaming.
    """
    if camera_obj.is_image_sequence:
        frame_bgr = camera_obj.get_frame(frame_number)
    else:
        video_files = [p for p in camera_obj.path.iterdir() if p.suffix.lower() == ".mp4"]
        if not video_files:
            raise FileNotFoundError(f"No .mp4 found in {camera_obj.path}")

        cap = cv2.VideoCapture(str(video_files[0]))
        try:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
            ok, frame_bgr = cap.read()
        finally:
            cap.release()   # always release -- this capture is throwaway

        if not ok:
            raise IndexError(f"Could not read frame {frame_number} from {video_files[0]}")

    if show:
        # plot_frame() is a Camera convenience method: converts BGR -> the
        # requested color space and titles the plot with the frame index
        # and its timestamp (when available).
        camera_obj.plot_frame(frame=frame_bgr, frame_number=frame_number, color=color)

    return frame_bgr


# Look at the very first frame of the flight before running anything downstream.
_ = open_frame(camera_obj, frame_number=10000)

### 4.2 — Reload the final result (`.parquet`)

`run_photogrammetry()` (Step 5) always writes its final merged DataFrame
into `OUTPUT_DIR/<flight_name>/` before returning it — so you never need to
re-run the whole pipeline just to look at results again later.
`<flight_name>` is resolved the same way internally by the library:
`flight.metadata["flight_name"]` if it's set, otherwise the name of the
flight's parent folder. `flight_result_dir()` below mirrors that exact logic
so the two never disagree.

The parquet's exact filename has varied across versions of `pils` — the
current `Camera.run_photogrammetry()` writes a fixed
`attitude_reconstruction.parquet`, but results produced by older versions
(or by the standalone `IPA_flight/workflows/photogrammetry.py` CLI) can be
named `attitude_reconstruction_FLY<video_code>.parquet` instead. Rather than
hard-code one filename, `load_photogrammetry_result()` below just globs for
any `*.parquet` file in that folder and picks the most recently modified
match, so it finds the right file regardless of naming convention.


In [ ]:
def flight_result_dir(flight: Flight, output_dir: str | Path) -> Path:
    """Per-flight output subfolder -- matches run_photogrammetry()'s own logic exactly."""
    flight_name = (
        flight.metadata.get("flight_name")
        or Path(flight.flight_info["drone_data_folder_path"]).parent.name
    )
    return Path(output_dir) / flight_name


def load_photogrammetry_result(flight: Flight, output_dir: str | Path) -> pl.DataFrame:
    """Reload the final merged pipeline output for `flight` without re-running it.

    Just globs for any `*.parquet` in the flight's result directory rather
    than assuming an exact filename -- see the note in 4.2 on why the name
    isn't fully stable across pils versions / the older standalone CLI. If
    more than one match exists, the most recently modified one wins.
    """
    result_dir = flight_result_dir(flight, output_dir)
    matches = sorted(
        result_dir.glob("*.parquet"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not matches:
        raise FileNotFoundError(
            f"No .parquet file found in {result_dir} -- run the pipeline first (Step 5)."
        )
    if len(matches) > 1:
        print(f"Multiple .parquet files found in {result_dir}; using the newest: {matches[0].name}")
        print(f"  (others: {[p.name for p in matches[1:]]})")
    return pl.read_parquet(matches[0])

### 4.3 — Reload an intermediate checkpoint (`dictionary_pN.ecsv`)

Every pipeline step writes its output DataFrame to
`OUTPUT_DIR/<flight_name>/dictionary_pN.ecsv` (`.ecsv` = Astropy's
self-describing CSV format) before moving on to the next step. These are
the exact same files `start_from_dict` (Step 6) reads from — inspecting
them here lets you debug one specific step without waiting for the whole
pipeline to finish.

| File                  | Produced by (step)                        | Contents                                    |
|-----------------------|--------------------------------------------|----------------------------------------------|
| `dictionary_p2.ecsv`  | `TargetFinder`                              | per-frame target pixel `(x, y)` positions      |
| `dictionary_p3.ecsv`  | `AttitudeReconstruction` (PnP / MCMC)       | camera `rvec` / `tvec` per frame               |
| `dictionary_p4.ecsv`  | `DroneData`                                 | drone GPS correlated to each frame            |
| `dictionary_p5.ecsv`  | `AttitudeReconstruction` (GPS correction)   | `tvec` refined against the drone GPS track    |
| `dictionary_p6.ecsv`  | `ConcerningTelescope`                       | telescope az/el + yaw/pitch/roll              |
| `dictionary_p7.ecsv`  | `PolarizationAngle`                         | attitude in the line-of-sight (LOS) frame     |


In [ ]:
from IPA_flight.IPA_flight.utils import load_dictionary

def load_checkpoint(flight: Flight, output_dir: str | Path, step: int) -> pl.DataFrame:
    """Reload `dictionary_p{step}.ecsv` (step in 2..7) as a polars DataFrame."""
    ecsv_path = flight_result_dir(flight, output_dir) / f"dictionary_p{step}.ecsv"
    return pl.from_pandas(load_dictionary(str(ecsv_path)))   # load_dictionary returns pandas

### 4.4 — Quick single-frame look (photogrammetry + EKF cross-check)

Sometimes you just want a fast sanity check on *one* frame — e.g. "does the
photogrammetric attitude roughly agree with the EKF here?" — without waiting
on a full Step 5 run or its interactive target-clicking.

This is intentionally narrower than the full pipeline:

- It reuses the **step p2 checkpoint** (`dictionary_p2.ecsv` — per-frame
  target pixel detections) instead of doing fresh detection/clicking, and
  runs just the step p3 solver (`AttitudeReconstruction.run_pnp`) scoped to
  one frame — `run_pnp` accepts any number of frames and degenerates
  correctly to a single `cv.solvePnP` call when given one. **Prerequisite:**
  Step 5 (or at least its step p2) must have already produced that
  checkpoint for this flight.
- There is no single-frame equivalent of steps p4-p7 — GPS correlation and
  the telescope/LOS frames are inherently defined over a *trajectory*, not
  one instant, so this quick look stops at the raw world→camera attitude
  (§3.1 of `REFERENCE_FRAMES_AND_MATH.md`), not the corrected/telescope-frame
  outputs of the full pipeline.
- The EKF comparison reloads an **already-computed** EKF solution via
  `pils.analyze.ekf.EKFAnalysis` (fusing IMU + photogrammetry quaternions
  through a Rust binary — a separate, heavier analysis in its own right) and
  looks up its nearest sample in time. **Prerequisite:** an EKF run must
  already have been saved for this flight (`EKFAnalysis(flight).run_analysis(
  imu_df, photo_df, save_data=True)`); this notebook does not build the
  `imu_df`/`photo_df` inputs that a fresh EKF run would need, since those are
  campaign/sensor-specific.
- The frame-to-EKF time match below is a coarse "nearest timestamp" lookup
  for a fast visual check — it does **not** apply any camera/EKF clock
  offset correction. For a rigorous alignment, see `find_time_offset` /
  `align_to_ekf` in `IPA_flight/IPA_flight/sensors.py`.


In [ ]:
from pils.analyze.ekf import EKFAnalysis


def load_ekf(flight: Flight) -> pl.DataFrame:
    """Reload the most recent saved EKF attitude solution for `flight`.

    EKFAnalysis fuses IMU + photogrammetry-quaternion data through a Rust EKF
    binary and persists results under
    `<flight_path>/proc/ekf/ekf_solution.h5`, versioned by run timestamp
    (`ekf.list_versions()` lists them all). This just reloads the newest one.

    Returns
    -------
    pl.DataFrame with columns: timestamp_s, timestamp_monotonic_ns,
    roll_deg, pitch_deg, yaw_deg, euler_cov_0_0 .. euler_cov_2_2.
    """
    ekf = EKFAnalysis(flight)
    version = ekf.get_latest_version()
    if version is None:
        raise FileNotFoundError(
            f"No saved EKF solution found in {ekf.ekf_dir} -- run "
            "EKFAnalysis(flight).run_analysis(imu_df, photo_df, save_data=True) first."
        )
    return version.ekf_data


def frame_timestamp(flight: Flight, camera_obj: Camera, frame_number: int) -> pd.Timestamp:
    """Absolute UTC timestamp of `frame_number`, for either Camera mode.

    - Alvium (image_sequence): looked up from the per-frame log DataFrame
      add_camera_data() already parsed (columns: timestamp, frame_num) --
      camera_obj.get_timestamp() always returns None for this mode.
    - Sony (video): computed by Camera.get_timestamp() from the recording
      start time (parsed from the .log file) plus frame_number / fps.
    """
    if camera_obj.is_image_sequence:
        cam_df = flight.raw_data.payload_data.camera
        row = cam_df.filter(pl.col("frame_num") == frame_number)
        if row.height == 0:
            raise ValueError(f"No log entry for frame {frame_number} in the Alvium log.")
        return pd.Timestamp(row["timestamp"][0], unit="s", tz="UTC")

    ts = camera_obj.get_timestamp(frame_number)
    if ts is None:
        raise RuntimeError(
            "camera_obj.tstart/fps not available -- can't compute a per-frame timestamp."
        )
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts

In [ ]:
from IPA_flight.IPA_flight.attitude import AttitudeReconstruction
from IPA_flight.IPA_flight.utils import rvec2euler


def run_photogrammetry_single_frame(
    flight: Flight,
    cfg: PhotogrammetryConfig,
    output_dir: str | Path,
    frame_number: int,
) -> dict:
    """Photogrammetric attitude for ONE frame -- a fast preview, not a pipeline run.

    Filters the step p2 checkpoint down to `frame_number` and runs the same
    PnP solver step p3 uses (AttitudeReconstruction.run_pnp), scoped to that
    single frame -- see the note in 4.4 on what this does and doesn't cover.
    """
    p2 = load_checkpoint(flight, output_dir, step=2)
    frame_df = p2.filter(pl.col("frame") == frame_number)
    if frame_df.height == 0:
        raise ValueError(
            f"Frame {frame_number} has no target detections in dictionary_p2.ecsv -- "
            "pick a frame inside the tracked scanning range, or run Step 5 first."
        )

    ar  = AttitudeReconstruction(cfg.camera_matrix, cfg.distortion_coeffs)
    # run_pnp accepts any number of frames; with only one frame's rows it
    # reshapes to a single (n_targets, 2)/(n_targets, 3) pair and runs exactly
    # one cv.solvePnP call -- so this is fast even though it's the "full" solver.
    pnp = ar.run_pnp(frame_df, ransac=cfg.pnp.get("ransac", False))
    row = pnp.iloc[0]   # run_pnp always returns a pandas DataFrame

    rvec = np.array([row["rvec_x"], row["rvec_y"], row["rvec_z"]])
    # return_enu=True -> Euler angles of the camera's orientation *in the
    # world/ENU frame* (i.e. the inverse of the raw world->camera rvec),
    # which is what's directly comparable to the EKF's roll/pitch/yaw below.
    roll, pitch, yaw = rvec2euler(rvec, return_enu=True, orientation="XYZ")

    return {
        "frame":     frame_number,
        "n_targets": frame_df.height,
        "rvec":      rvec,
        "quat":      np.array([row["quat_x"], row["quat_y"], row["quat_z"], row["quat_w"]]),
        "roll_deg":  roll,
        "pitch_deg": pitch,
        "yaw_deg":   yaw,
    }

In [ ]:
def quick_look(
    flight: Flight,
    camera_obj: Camera,
    cfg: PhotogrammetryConfig,
    output_dir: str | Path,
    frame_number: int,
) -> dict:
    """One-stop preview of a single frame: image + photogrammetric attitude +
    nearest EKF attitude, for a fast sanity check without a full pipeline run.
    """
    print(f"--- Frame {frame_number} ---")
    _ = open_frame(camera_obj, frame_number)   # shows the image (Step 4.1)

    photo = run_photogrammetry_single_frame(flight, cfg, output_dir, frame_number)
    print(
        f"Photogrammetry (n={photo['n_targets']} targets): "
        f"roll={photo['roll_deg']:.2f}, pitch={photo['pitch_deg']:.2f}, "
        f"yaw={photo['yaw_deg']:.2f} deg"
    )

    try:
        ekf_df = load_ekf(flight).to_pandas()
        t_frame = frame_timestamp(flight, camera_obj, frame_number).timestamp()
        idx = (ekf_df["timestamp_s"] - t_frame).abs().idxmin()
        nearest = ekf_df.loc[idx]
        print(
            f"Nearest EKF sample (Δt={nearest['timestamp_s'] - t_frame:+.3f}s): "
            f"roll={nearest['roll_deg']:.2f}, pitch={nearest['pitch_deg']:.2f}, "
            f"yaw={nearest['yaw_deg']:.2f} deg"
        )

        plt.figure()
        plt.title(f"Frame {frame_number}: Photogrammetry vs EKF")
        plt.plot(
            [0, 1],
            [photo["roll_deg"], nearest["roll_deg"]],
            marker="o",
            label="roll",
        )
        plt.plot(
            [0, 1],
            [photo["pitch_deg"], nearest["pitch_deg"]],
            marker="o",
            label="pitch",
        )
        plt.plot(
            [0, 1],
            [photo["yaw_deg"], nearest["yaw_deg"]],
            marker="o",
            label="yaw",
        )
        plt.xticks([0, 1], ["Photogrammetry", "EKF"])
        plt.ylabel("Degrees")
        plt.legend()
        plt.grid()

    except FileNotFoundError as e:
        print(f"(no EKF comparison available: {e})")

    return photo


# Quick look at the first tracked frame -- requires dictionary_p2.ecsv to
# already exist for this flight (Step 5, or at least its step p2, already run).
if "dictionary_p2.ecsv" in [p.name for p in flight_result_dir(flight, OUTPUT_DIR).iterdir()]:
    _ = quick_look(flight, camera_obj, cfg, OUTPUT_DIR, frame_number=10000)

--- Frame 10000 ---


Running PnP attitude estimation: 100%|██████████| 1/1 [00:00<00:00, 3650.40it/s]
2026-09-10 15:35:46,073 - pils.analyze.ekf - INFO - Initialized EKF analysis for flight: /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530
2026-09-10 15:35:46,073 - pils.analyze.ekf - INFO - EKF directory: /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/proc/ekf


Photogrammetry (n=35 targets): roll=-140.62, pitch=2.67, yaw=-86.23 deg
(no EKF comparison available: No saved EKF solution found in /data/POLOCALC/campaigns/202511/20251206/flight_20251206_1530/proc/ekf -- run EKFAnalysis(flight).run_analysis(imu_df, photo_df, save_data=True) first.)


## 5 — Run the full pipeline from scratch

`camera_obj.run_photogrammetry(...)` orchestrates six sequential steps, each
one consuming the output of the previous one and writing its own
`dictionary_pN.ecsv` checkpoint (see 4.3) before moving on:

| Step | Class.method                                       | What it does |
|------|------------------------------------------------------|--------------|
| p2   | `TargetFinder.finderSequential`                       | Tracks the surveyed targets' pixel `(x, y)` across frames. **Interactive** by default (`finder.tg_by_hand="click_with_tel"` in the config): a matplotlib window opens for you to click telescopes then targets once — the rest is tracked automatically frame-to-frame. |
| p3   | `AttitudeReconstruction.run_pnp` (or `.run_mcmc`)     | Solves camera attitude (`rvec`/`tvec`) per frame via PnP (fast, default) or full MCMC (slow, more robust to outliers — set `mcmc_solution=True`). |
| p4   | `DroneData.correlate_drone_photo`                     | Aligns each frame with the closest drone GPS fix by timestamp. |
| p5   | `AttitudeReconstruction._correct_tvec_with_gps`       | Refines `tvec` using the drone GPS track as an extra constraint. |
| p6   | `ConcerningTelescope.drone_in_telescope_frame`        | Expresses drone position/attitude in each telescope's own frame (az/el, yaw/pitch/roll). |
| p7   | `PolarizationAngle.drone_attitude_in_LOS_frame`       | Final attitude in the telescope's line-of-sight (LOS) frame -- the polarization-angle-ready output. |

The six steps' results are merged on `frame` and written to an
`attitude_reconstruction*.parquet` file in `OUTPUT_DIR/<flight_name>/` (see
4.2 for why the exact filename isn't 100% fixed across versions) before
being returned.

**This cell is interactive** — a plot window opens during step p2 asking you
to click on targets, unless you pass `start_from_dict` to skip past it (see
Step 6 for resuming instead of running from scratch).


In [ ]:
result = camera_obj.run_photogrammetry(
    csv_file=TARGETS_CSV,
    config=cfg,
    flight=flight,
    output_dir=OUTPUT_DIR,
    check_results=None,      # None -> auto-saves diagnostic plots to <output>/<flight>/plots/
    start_from_dict=None,    # None -> full run, starting from step p2
    mcmc_solution=False,     # True -> use run_mcmc instead of run_pnp for step p3 (much slower)  
    
    target_indices=None,     # e.g. [0, 2, 4] to only use a subset of the surveyed targets
)

print(f"Result: {result.shape[0]} frames x {result.shape[1]} columns")
result.head()

2026-09-10 15:35:46,185 - pils.sensors.camera - INFO - Photogrammetry output dir: /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530
2026-09-10 15:35:46,188 - IPA_flight.IPA_flight.genParamFile - INFO - DJI base (antenna head): [ -22.95977493  -67.78669737 5182.496     ]
2026-09-10 15:35:46,189 - IPA_flight.IPA_flight.genParamFile - INFO - DJI base self-measured (avg 5 rows): [ -22.9598223   -67.78685073 5182.1728    ]
2026-09-10 15:35:46,189 - IPA_flight.IPA_flight.genParamFile - INFO - RTK base correction:
  Surveyed base : [ -22.95977493  -67.78669737 5182.496     ]
  Logged base   : [ -22.9597732  -67.7866847 5173.02     ]
  GPS offset    : [-1.729999997479581e-06, -1.2670000003822679e-05, 9.475999999999658]
2026-09-10 15:35:46,189 - IPA_flight.IPA_flight.genParamFile - INFO - GPS offset [dlat, dlon, dalt]: [-1.729999997479581e-06, -1.2670000003822679e-05, 9.475999999999658]
2026-09-10 15:35:46,191 - IPA_flight.IPA_flight.genParamFile - INFO - Geodetic targets: 52 po

Frame rate:  29.97002997002997
📹 Streaming video from frame 5865 → 15365

🖱  [INIT] Click telescopes (cyan), then targets (red).
   RIGHT-click = place | SPACE = skip | Z = undo | ENTER = finish

✅ Telescopes done — now click the science TARGETS (red)
  → 4 telescope GCPs collected
  → 0/52 targets clicked

📐 Running PnP from 4 telescope GCPs …
🔁 Iterative target refinement …
  Iteration 0: mean pixel shift = 3.121 px
  Iteration 1: mean pixel shift = 0.444 px
  Iteration 2: mean pixel shift = 0.000 px
  Iteration 3: mean pixel shift = 0.000 px
  ✅ Converged


/home/fastori/anaconda3/envs/photo/lib/python3.12/site-packages/IPA_flight/IPA_flight/finder.py:812: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


  ✖ Target 34 removed.
  ✖ Target 35 removed.
  ✖ Target 36 removed.
  ✖ Target 37 removed.
✖ Dropping 4 removed target(s): [34, 35, 36, 37]
✅ Initialization accepted.
         x       y      R      G      B
0   1950.0  1328.0  125.0  157.0  210.0
1   1846.0  1200.0  108.0  141.0  201.0
2   1808.0  1380.0  118.0  144.0  190.0
3   1858.0  1280.0  122.0  148.0  194.0
4   2170.0  1228.0  129.0  146.0  189.0
5   1992.0   848.0  131.0  151.0  195.0
6   2256.0   902.0  132.0  148.0  177.0
7   2316.0  1024.0  144.0  152.0  182.0
8   2214.0  1024.0  143.0  157.0  196.0
9   2120.0   944.0  136.0  150.0  189.0
10  2148.0  1840.0  125.0  148.0  175.0
11  2040.0  1956.0  100.0  140.0  172.0
12  2006.0  1788.0  114.0  143.0  186.0
13  1938.0  1878.0   99.0   82.0   85.0
14  1846.0  1840.0  108.0  145.0  183.0
15  1710.0  1838.0  110.0  143.0  187.0
16  1730.0  1726.0  109.0  154.0  194.0
17  1622.0  1754.0  109.0  150.0  193.0
18  1564.0  1998.0   94.0  141.0  189.0
19  1486.0  1700.0   90.0  144.0

Tracking frames: 100%|█████████▉| 9499/9500 [05:07<00:00, 30.86it/s]


...

2026-09-10 15:45:33,610 - pils.sensors.camera - INFO - Photogrammetry step: dictionary_p6.ecsv
2026-09-10 15:45:36,355 - pils.sensors.camera - INFO - Photogrammetry step: dictionary_p7.ecsv


✅ Saved 4×5 telescope attitude grid plot to /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530/plots/telescope_pointing_dictionary_p6.ecsv.jpg
✅ Saved 4×5 telescope attitude grid plot to /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530/plots/telescope_pointing_dictionary_p7.ecsv.jpg
✅ Saved LOS-frame attitude plot to /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530/plots/telescope_attitude_dictionary_p7.ecsv.jpg


2026-09-10 15:45:43,994 - pils.sensors.camera - INFO - Photogrammetry result saved: /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530/attitude_reconstruction.parquet


Result: 1852500 frames x 41 columns


frame,target_id,x,y,target_E,target_N,target_U,quat_x,quat_y,quat_z,quat_w,rvec_x,rvec_y,rvec_z,tvec_x,tvec_y,tvec_z,time,tvec_E,tvec_N,tvec_U,drone_E,drone_N,drone_U,quat_x_corr,quat_y_corr,quat_z_corr,quat_w_corr,rvec_x_corr,rvec_y_corr,rvec_z_corr,projection_error,tel_name,az,el,yaw,pitch,roll,yaw_LOS,pitch_LOS,roll_LOS
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64
5866,0,1949.873552,1328.874542,-110.967835,-52.024751,2.160823,0.676724,0.638819,0.264873,-0.252579,1.840058,1.736992,0.720208,49.267036,156.169302,519.351355,1.7650e9,-176.651245,-380.520018,347.189826,-177.149528,-379.610299,346.850738,0.676701,0.639085,0.265146,-0.251684,1.840844,1.738516,0.721281,2.275716,"""SATp1""",-179.277149,44.982885,-86.671835,42.886085,179.864864,-88.090086,-2.100316,-178.291429
5866,0,1949.873552,1328.874542,-110.967835,-52.024751,2.160823,0.676724,0.638819,0.264873,-0.252579,1.840058,1.736992,0.720208,49.267036,156.169302,519.351355,1.7650e9,-176.651245,-380.520018,347.189826,-177.149528,-379.610299,346.850738,0.676701,0.639085,0.265146,-0.251684,1.840844,1.738516,0.721281,2.275716,"""SATp2""",-175.463784,44.879629,-86.671835,42.886085,179.864864,-90.885788,-2.227669,179.008234
5866,0,1949.873552,1328.874542,-110.967835,-52.024751,2.160823,0.676724,0.638819,0.264873,-0.252579,1.840058,1.736992,0.720208,49.267036,156.169302,519.351355,1.7650e9,-176.651245,-380.520018,347.189826,-177.149528,-379.610299,346.850738,0.676701,0.639085,0.265146,-0.251684,1.840844,1.738516,0.721281,2.275716,"""SATp3""",-172.156282,43.653696,-86.671835,42.886085,179.864864,-93.312716,-3.365769,176.593712
5866,0,1949.873552,1328.874542,-110.967835,-52.024751,2.160823,0.676724,0.638819,0.264873,-0.252579,1.840058,1.736992,0.720208,49.267036,156.169302,519.351355,1.7650e9,-176.651245,-380.520018,347.189826,-177.149528,-379.610299,346.850738,0.676701,0.639085,0.265146,-0.251684,1.840844,1.738516,0.721281,2.275716,"""CLASS1""",-163.889818,53.104581,-86.671835,42.886085,179.864864,-99.392331,6.619045,172.18051
5866,0,1949.873552,1328.874542,-110.967835,-52.024751,2.160823,0.676724,0.638819,0.264873,-0.252579,1.840058,1.736992,0.720208,49.267036,156.169302,519.351355,1.7650e9,-176.651245,-380.520018,347.189826,-177.149528,-379.610299,346.850738,0.676701,0.639085,0.265146,-0.251684,1.840844,1.738516,0.721281,2.275716,"""CLASS2""",-163.078942,40.666625,-86.671835,42.886085,179.864864,-99.963071,-5.550445,169.54712


# 5.1 Telescope Driven Analysis
    - Attempt to parallelize the target tracking pipeline, not yet working

In [ ]:
# # ── Test finderTelescopeDriven as a drop-in replacement for step p2 ────────
# from IPA_flight.IPA_flight.finder import TargetFinder
# from IPA_flight.IPA_flight.genParamFile import GenParamFile

# # Same setup run_photogrammetry does internally to build a TargetFinder --
# # duplicated here since we're calling the tracker directly, before handing
# # its output back into the normal pipeline via precomputed_targets.
# params = GenParamFile.GeoPlot(
#     csv_file=str(TARGETS_CSV),
#     reference_point=cfg.reference_point,
#     flight=flight,
#     dji_base_logged=getattr(cfg, "dji_base_logged", None),
# )
# dh_wrapper = Camera._build_dh_wrapper(flight, camera_obj)

# target_finder = TargetFinder(
#     dh_wrapper,
#     params["geodetic_targets"],
#     params["geodetic_tel_positions"],
#     params["image_tel_positions"],
#     cfg.camera_matrix,
#     cfg.distortion_coeffs,
#     params["reference_point"],
#     params["video_code"],
# )

# # box_width/box_height/tracking_method/refine_threshold mirror cfg.finder --
# # passed explicitly since finderTelescopeDriven doesn't take tg_by_hand/color_tol.
# telescope_targets_df = target_finder.finderTelescopeDriven(
#     box_width=cfg.finder.get("box_width", 10),
#     box_height=cfg.finder.get("box_height", 10),
#     tracking_method=cfg.finder.get("tracking_method", "optical_flow"),
#     orb_nfeatures=cfg.finder.get("orb_nfeatures", 10000),
#     refine_threshold=cfg.finder.get("refine_threshold", 0.98),
#     min_telescopes=4,
# )

# print(f"Step p2 (telescope-driven): {telescope_targets_df.shape[0]} rows, "
#       f"{telescope_targets_df['target_id'].nunique()} targets")

# # ── Feed straight into the normal pipeline (p3 -> p7 unchanged) ────────────
# # NOTE: point output_dir at a separate folder if you want to keep the
# # existing dictionary_p2.ecsv/attitude_reconstruction.parquet for comparison
# # -- save_intermediate_results will otherwise overwrite them in place.
# result_telescope = camera_obj.run_photogrammetry(
#     csv_file=TARGETS_CSV,
#     config=cfg,
#     flight=flight,
#     output_dir=OUTPUT_DIR,
#     check_results=None,
#     start_from_dict=None,
#     precomputed_targets=telescope_targets_df,   # <-- skips finderSequential, uses ours instead
#     mcmc_solution=False,
#     target_indices=None,
# )

# print(f"Result: {result_telescope.shape[0]} frames x {result_telescope.shape[1]} columns")
# result_telescope.head()

## 6 — Resume from a checkpoint instead

If a run was interrupted, or you only want to re-run steps p4 onward (e.g.
after tweaking `drone_correlation` in the YAML), point `start_from_dict` at
the last good checkpoint from 4.3 instead of redoing Step 5. Step p2 (the
interactive one) and every step before the checkpoint are skipped entirely
— previously clicked targets are reused as-is.


In [ ]:
checkpoint = flight_result_dir(flight, OUTPUT_DIR) / "dictionary_p3.ecsv"

result = camera_obj.run_photogrammetry(
    csv_file=TARGETS_CSV,
    config=cfg,
    flight=flight,
    output_dir=OUTPUT_DIR,
    start_from_dict=checkpoint,   # resumes at step p3, reusing p2 as-is
)

result.head()

2026-09-10 15:45:46,690 - pils.sensors.camera - INFO - Photogrammetry output dir: /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530
2026-09-10 15:45:46,693 - IPA_flight.IPA_flight.genParamFile - INFO - DJI base (antenna head): [ -22.95977493  -67.78669737 5182.496     ]
2026-09-10 15:45:46,694 - IPA_flight.IPA_flight.genParamFile - INFO - DJI base self-measured (avg 5 rows): [ -22.9598223   -67.78685073 5182.1728    ]
2026-09-10 15:45:46,695 - IPA_flight.IPA_flight.genParamFile - INFO - RTK base correction:
  Surveyed base : [ -22.95977493  -67.78669737 5182.496     ]
  Logged base   : [ -22.9597732  -67.7866847 5173.02     ]
  GPS offset    : [-1.729999997479581e-06, -1.2670000003822679e-05, 9.475999999999658]
2026-09-10 15:45:46,695 - IPA_flight.IPA_flight.genParamFile - INFO - GPS offset [dlat, dlon, dalt]: [-1.729999997479581e-06, -1.2670000003822679e-05, 9.475999999999658]
2026-09-10 15:45:46,696 - IPA_flight.IPA_flight.genParamFile - INFO - Geodetic targets: 52 po

Frame rate:  29.97002997002997


Correlating drone log and photogrammetry: 100%|██████████| 16341/16341 [00:00<00:00, 40750.19it/s]


Time alignment: correlation (index 9467).


2026-09-10 15:45:47,915 - pils.sensors.camera - INFO - Photogrammetry step: dictionary_p5.ecsv


Saved plot to /home/fastori/Desktop/photogrammetry_results/flight_20251206_1530/plots/drone_gps_dictionary_p4.ecsv.jpg
Median offset between GPS and Photo distances: 0.734 m


KeyboardInterrupt: 

## 7 — Inspect the result

Continue with `result` from Step 5 or 6 above, or reload a previous run
without re-running anything using the Step 4.2 helper:


In [ ]:
def plot_raw_detections(camera_obj: Camera, flight: Flight, output_dir: str | Path,
                         dict_name: str = "dictionary_p2.ecsv",
                         compare_dict_name: str | None = None,
                         frame_number: int | None = None):
    """Plot a step-p2-stage dictionary's raw x,y detections directly on the
    video frame -- bypasses PnP/GPS correction entirely, a pure sanity check
    on tracking alone.

    dict_name, compare_dict_name : str
        Dictionary filename(s) under flight_result_dir(flight, output_dir),
        e.g. "dictionary_p2.ecsv" / "dictionary_p2_PW.ecsv". Pass
        compare_dict_name to overlay a second file's detections (red) on top
        of the first (lime) for a direct visual diff.
    frame_number : int | None
        Frame to plot. None (default) uses the last tracked frame in
        dict_name.
    """
    from IPA_flight.IPA_flight.utils import load_dictionary

    def _load(name):
        path = flight_result_dir(flight, output_dir) / name
        df = load_dictionary(str(path))
        return (df if isinstance(df, pl.DataFrame) else pl.from_pandas(df)), path

    df, path = _load(dict_name)
    if frame_number is None:
        frame_number = int(df["frame"].max())

    frame_df = df.filter(pl.col("frame") == frame_number).sort("target_id")
    if frame_df.height == 0:
        raise ValueError(f"Frame {frame_number} not found in {path}")

    frame_bgr = open_frame(camera_obj, frame_number=frame_number, show=False)

    fig, ax = plt.subplots(figsize=(14, 9))
    ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

    x, y, tids = frame_df["x"].to_numpy(), frame_df["y"].to_numpy(), frame_df["target_id"].to_list()
    ax.scatter(x, y, s=140, facecolors="none", edgecolors="lime", linewidths=2, label=dict_name)
    for tid, xi, yi in zip(tids, x, y):
        ax.annotate(str(int(tid)), (xi, yi), xytext=(8, -10), textcoords="offset points",
                    color="yellow", fontsize=9)

    if compare_dict_name:
        cdf, cpath = _load(compare_dict_name)
        cframe_df = cdf.filter(pl.col("frame") == frame_number).sort("target_id")
        if cframe_df.height == 0:
            print(f"⚠️ Frame {frame_number} not found in {cpath} -- skipping overlay.")
        else:
            cx, cy, ctids = cframe_df["x"].to_numpy(), cframe_df["y"].to_numpy(), cframe_df["target_id"].to_list()
            ax.scatter(cx, cy, marker="x", s=100, color="red", linewidths=2, label=compare_dict_name)
            for tid, xi, yi in zip(ctids, cx, cy):
                ax.annotate(str(int(tid)), (xi, yi), xytext=(8, 6), textcoords="offset points",
                            color="red", fontsize=9)

    ax.set_title(f"Raw tracker detections -- frame {frame_number}")
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"Frame {frame_number}: {frame_df.height} targets from {dict_name}"
          + (f", {cframe_df.height if compare_dict_name and cframe_df.height else 0} from {compare_dict_name}" if compare_dict_name else ""))
    return frame_number

def plot_gps_correction(result: pl.DataFrame, tel_name: str):
    """Scatter the drone's ENU position before vs. after GPS correction.

    tel_name : one of `result["tel_name"].unique()` -- selects which
        telescope's rows to use (see 7.7's note on why that's needed;
        the plotted quantities themselves don't depend on the telescope).
    """
    required = ["tel_name", "tvec_E", "tvec_N", "tvec_U", "drone_E", "drone_N", "drone_U"]
    if not all(c in result.columns for c in required):
        print("GPS-correction columns not found -- pipeline may not have reached step p4/p6.")
        return

    df = (
        result
        .filter(pl.col("tel_name") == tel_name)
        .select(["frame"] + required[1:])
        .drop_nulls()
        .unique(subset=["frame"])
        .sort("frame")
    )
    if df.height == 0:
        raise ValueError(f"No rows with tel_name={tel_name!r} and both tvec_*/drone_* populated.")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].scatter(df["tvec_E"], df["tvec_N"], s=10, alpha=0.5, color="steelblue", label="Before (photogrammetry tvec)")
    axes[0].scatter(df["drone_E"], df["drone_N"], s=10, alpha=0.5, color="orange", label="After (GPS drone position)")
    axes[0].set_xlabel("East (m)")
    axes[0].set_ylabel("North (m)")
    axes[0].set_title(f"Ground track -- {tel_name}")
    axes[0].set_aspect("equal")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(df["frame"], df["tvec_U"], s=10, alpha=0.5, color="steelblue", label="Before (tvec_U)")
    axes[1].scatter(df["frame"], df["drone_U"], s=10, alpha=0.5, color="orange", label="After (drone_U)")
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Up / altitude (m)")
    axes[1].set_title(f"Altitude -- {tel_name}")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    offset = np.sqrt(
        (df["tvec_E"] - df["drone_E"]) ** 2
        + (df["tvec_N"] - df["drone_N"]) ** 2
        + (df["tvec_U"] - df["drone_U"]) ** 2
    )
    print(f"Median |before - after| offset: {offset.median():.2f} m  (max {offset.max():.2f} m, n={df.height} frames)")

def plot_reprojected_targets(camera_obj: Camera, cfg: PhotogrammetryConfig,
                              result: pl.DataFrame, frame_number: int | None = None):
    """Reproject each target's known 3D position through the solved pose
    for one frame, and overlay it against the actually-tracked detection.

    frame_number : int | None
        Frame to check. None (default) picks a random frame that has a
        complete solved pose + target geometry.
    """
    rvec_cols = ["rvec_x_corr", "rvec_y_corr", "rvec_z_corr"]
    corrected = all(c in result.columns for c in rvec_cols)
    if not corrected:
        rvec_cols = ["rvec_x", "rvec_y", "rvec_z"]

    pose_cols = rvec_cols + (["drone_E", "drone_N", "drone_U"] if corrected else ["tvec_x", "tvec_y", "tvec_z"])
    target_cols = ["target_id", "target_E", "target_N", "target_U", "x", "y"]

    df = result.select(["frame"] + pose_cols + target_cols)

    if frame_number is None:
        candidates = df.drop_nulls(subset=["frame"] + pose_cols)["frame"].unique().to_list()
        if not candidates:
            raise ValueError("No frame with a complete solved pose found.")
        frame_number = int(np.random.choice(candidates))

    frame_df = df.filter(pl.col("frame") == frame_number).drop_nulls(subset=pose_cols)
    if frame_df.height == 0:
        raise ValueError(f"Frame {frame_number} has no rows with a complete pose.")

    pose_row = frame_df.row(0, named=True)
    rvec = np.array([pose_row[c] for c in rvec_cols], dtype=np.float64)

    if corrected:
        # Mirrors AttitudeReconstruction._correct_tvec_with_gps: tvec is
        # re-derived from the corrected rotation + the fixed GPS position,
        # since p5 doesn't save a tvec_corr column of its own.
        R_mat, _   = cv2.Rodrigues(rvec)
        drone_pos  = np.array([pose_row["drone_E"], pose_row["drone_N"], pose_row["drone_U"]])
        tvec       = -(R_mat @ drone_pos.reshape(3, 1)).flatten()
    else:
        tvec = np.array([pose_row["tvec_x"], pose_row["tvec_y"], pose_row["tvec_z"]])

    targets       = frame_df.select(target_cols).unique(subset=["target_id"]).sort("target_id")
    object_points = targets.select(["target_E", "target_N", "target_U"]).to_numpy()
    detected_px   = targets.select(["x", "y"]).to_numpy()
    target_ids    = targets["target_id"].to_list()

    reprojected, _ = cv2.projectPoints(
        object_points, rvec.reshape(3, 1), tvec.reshape(3, 1),
        cfg.camera_matrix, cfg.distortion_coeffs,
    )
    reprojected = reprojected.reshape(-1, 2)

    frame_bgr = open_frame(camera_obj, frame_number=frame_number, show=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    ax.scatter(detected_px[:, 0], detected_px[:, 1], s=120, facecolors="none",
               edgecolors="lime", linewidths=2, label="Detected (tracked)")
    ax.scatter(reprojected[:, 0], reprojected[:, 1], marker="x", s=100, color="red",
               linewidths=2, label=f"Reprojected ({'GPS-corrected' if corrected else 'raw PnP'})")
    for (dx, dy), (rx, ry) in zip(detected_px, reprojected):
        if not (np.isnan(dx) or np.isnan(dy)):
            ax.plot([dx, rx], [dy, ry], color="yellow", lw=1, alpha=0.6)
    for tid, (rx, ry) in zip(target_ids, reprojected):
        ax.annotate(str(int(tid)), (rx, ry), xytext=(8, -12),
                    textcoords="offset points", color="red", fontsize=10)

    ax.set_title(f"Frame {frame_number} -- reprojection check "
                 f"({'GPS-corrected' if corrected else 'raw PnP'} pose)")
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    valid = ~np.isnan(detected_px).any(axis=1)
    if valid.any():
        err = np.linalg.norm(detected_px[valid] - reprojected[valid], axis=1)
        print(f"Frame {frame_number}: reprojection error mean {err.mean():.2f} px, max {err.max():.2f} px")
    else:
        print(f"Frame {frame_number}: no detected targets to compare against (only reprojection shown).")

    return frame_number

def plot_tracking_grid(camera_obj: Camera, flight: Flight, output_dir: str | Path,
                        dict_name: str = "dictionary_p2.ecsv",
                        frame_step: int = 30, n_panels: int = 30, cols: int = 6,
                        mode: str = "even"):
    """
    Debug view: tile detections from `n_panels` sampled frames of a step-p2
    dictionary, to spot where/why the tracker drops targets.

    mode : {"even", "worst"}
        "even"  (default): frames spaced `frame_step` apart, starting at the
                 dictionary's first tracked frame -- a general scan.
        "worst": the `n_panels` frames with the most missing (NaN) targets --
                 jump straight to the problem regions instead of scanning
                 uniformly.
    """
    from IPA_flight.IPA_flight.utils import load_dictionary

    path = flight_result_dir(flight, output_dir) / dict_name
    df = load_dictionary(str(path))
    df = df if isinstance(df, pl.DataFrame) else pl.from_pandas(df)

    n_targets_total = df["target_id"].n_unique()

    # missing-target count per frame (NaN x)
    per_frame = (
        df.with_columns(pl.col("x").is_nan().alias("missing"))
          .group_by("frame")
          .agg(pl.col("missing").sum().alias("n_missing"))
          .sort("frame")
    )

    if mode == "even":
        all_frames = per_frame["frame"].to_list()
        frames = all_frames[::frame_step][:n_panels]
    elif mode == "worst":
        frames = per_frame.sort("n_missing", descending=True)["frame"].to_list()[:n_panels]
        frames.sort()
    else:
        raise ValueError(f"Unknown mode {mode!r} -- expected 'even' or 'worst'.")

    rows = int(np.ceil(len(frames) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.6))
    axes = np.atleast_2d(axes)

    for ax, frame_number in zip(axes.flat, frames):
        frame_df    = df.filter(pl.col("frame") == frame_number)
        x, y        = frame_df["x"].to_numpy(), frame_df["y"].to_numpy()
        missing_ids = frame_df.filter(pl.col("x").is_nan())["target_id"].to_list()

        frame_bgr = open_frame(camera_obj, frame_number=frame_number, show=False)
        ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        valid = ~np.isnan(x)
        ax.scatter(x[valid], y[valid], s=30, facecolors="none", edgecolors="lime", linewidths=1.2)

        title_color = "red" if missing_ids else "black"
        ax.set_title(
            f"f{frame_number} — missing {len(missing_ids)}/{n_targets_total}"
            + (f"\n{missing_ids}" if missing_ids else ""),
            fontsize=7, color=title_color,
        )
        ax.axis("off")

    for ax in axes.flat[len(frames):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("Worst frames by missing-target count:")
    print(per_frame.filter(pl.col("n_missing") > 0).sort("n_missing", descending=True).head(20))




In [ ]:
# Pick a telescope to inspect -- change this to compare a different one.
tel_name = result["tel_name"].drop_nulls().unique().sort()[1]
print(f"Using tel_name={tel_name!r} (available: {result['tel_name'].drop_nulls().unique().sort().to_list()})")

plot_gps_correction(result, tel_name=tel_name)




Using tel_name='CLASS2' (available: ['CLASS1', 'CLASS2', 'SATp1', 'SATp2', 'SATp3'])
Median |before - after| offset: 0.91 m  (max 2.32 m, n=9500 frames)


In [ ]:
_ = plot_raw_detections(camera_obj, flight, OUTPUT_DIR,
                         dict_name="dictionary_p2.ecsv",
                         compare_dict_name=None,  # set to None for single-file view
                         frame_number=10000)  # None = last tracked frame


Frame 10000: 38 targets from dictionary_p2.ecsv


In [ ]:
_ = plot_reprojected_targets(camera_obj, cfg, result, frame_number=15365)


Frame 15365: reprojection error mean 1.15 px, max 2.51 px


In [ ]:

# Switch mode="worst" once you want to zoom straight into the frames that are
# actually losing targets, instead of an even scan across the whole video.
plot_tracking_grid(camera_obj, flight, OUTPUT_DIR, dict_name="dictionary_p2.ecsv",
                    frame_step=1, n_panels=30, cols=6, mode="even")



Worst frames by missing-target count:
shape: (0, 2)
┌───────┬───────────┐
│ frame ┆ n_missing │
│ ---   ┆ ---       │
│ i64   ┆ u32       │
╞═══════╪═══════════╡
└───────┴───────────┘


### 7.1 — Column overview

Columns grouped by the pipeline step that produced them (see the table in
Step 5).


In [ ]:
groups = {
    "p2 - image targets":   [c for c in result.columns if c in ["frame", "x", "y", "target_id"]],
    "p3 - PnP attitude":     [c for c in result.columns if c.startswith(("rvec", "tvec", "quat", "proj"))],
    "p4 - drone GPS":        [c for c in result.columns if c.startswith(("drone_", "time"))],
    "p5 - GPS-corrected":    [c for c in result.columns if c.endswith("_corr") or "projection_error" in c],
    "p6 - telescope frame":  [c for c in result.columns if c in ["tel_name", "az", "el"]],
    "p7 - LOS frame":        [c for c in result.columns if "LOS" in c],
}

for group, cols in groups.items():
    if cols:
        print(f"\n{group}")
        print("  ", cols)

### 7.2 — Projection error (PnP quality check)

Reprojecting the solved 3-D targets back onto the image and comparing to
the tracked pixel positions gives a per-frame error in pixels — below
roughly 2-3 px generally means the PnP solution for that frame is reliable.


In [ ]:
pe = result.select(["frame", "time", "projection_error"]).drop_nulls()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pe["time"].to_numpy(), pe["projection_error"].to_numpy(), lw=1)
ax.axhline(2.0, color="orange", ls="--", label="2 px threshold")
ax.axhline(5.0, color="red",    ls="--", label="5 px threshold")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Projection error (px)")
ax.set_title("PnP projection error per frame")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Median: {pe['projection_error'].median():.2f} px")
print(f"Max:    {pe['projection_error'].max():.2f} px")
print(f"Frames > 5px: {(pe['projection_error'] > 5).sum()}")

### 7.3 — Attitude over time (roll / pitch / yaw)


In [ ]:
rvec_cols = ["rvec_x_corr", "rvec_y_corr", "rvec_z_corr"]
if not all(c in result.columns for c in rvec_cols):
    rvec_cols = ["rvec_x", "rvec_y", "rvec_z"]   # fall back to raw PnP if GPS correction (p5) wasn't run

att = result.select(["time_x"] + rvec_cols).drop_nulls()
t   = att["time_x"].to_numpy()
rv  = att.select(rvec_cols).to_numpy()   # (N, 3) Rodrigues vectors

angles_deg = np.degrees(rv)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, label, series in zip(axes, ["Roll (deg)", "Pitch (deg)", "Yaw (deg)"], angles_deg.T):
    ax.plot(t, series, lw=1)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
axes[0].set_title("Camera attitude (rvec, GPS-corrected if available)")
plt.tight_layout()
plt.show()

### 7.4 — Drone position in ENU (East / North / Up)


In [ ]:
pos = result.select(["time_x", "drone_E_x", "drone_N_x", "drone_U_x"]).drop_nulls()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pos["drone_E_x"].to_numpy(), pos["drone_N_x"].to_numpy(), lw=1)
axes[0].set_xlabel("East (m)")
axes[0].set_ylabel("North (m)")
axes[0].set_title("Ground track (ENU)")
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

axes[1].plot(pos["time_x"].to_numpy(), pos["drone_U_x"].to_numpy(), lw=1, color="steelblue")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Up / altitude (m)")
axes[1].set_title("Altitude over time")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.5 — Attitude in the telescope line-of-sight (LOS) frame

Final output of step p7 — drone attitude expressed relative to each
telescope's boresight, ready for polarization-angle analysis.


In [ ]:
los_cols = ["time_x", "yaw_LOS_x", "pitch_LOS_x", "roll_LOS_x"]
if all(c in result.columns for c in los_cols):
    los = result.select(los_cols).drop_nulls()
    t   = los["time"].to_numpy()

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for ax, col, label in zip(axes, ["yaw_LOS_x", "pitch_LOS_x", "roll_LOS_x"],
                               ["Yaw LOS (deg)", "Pitch LOS (deg)", "Roll LOS (deg)"]):
        ax.plot(t, np.degrees(los[col].to_numpy()), lw=1)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Time (s)")
    axes[0].set_title("Drone attitude in line-of-sight frame")
    plt.tight_layout()
    plt.show()
else:
    print("LOS columns not found -- pipeline may not have reached step p7.")

### 7.6 — Visual sanity check: worst-projection-error frame

Combine the `open_frame()` helper from 4.1 with the result table to look at
the actual image for the frame with the worst PnP fit — a quick way to spot
mis-tracked targets or motion blur.


In [ ]:
worst = result.sort("projection_error", descending=True).row(0, named=True)
print(f"Frame {worst['frame']} -- projection error {worst['projection_error']:.2f} px")

_ = open_frame(camera_obj, frame_number=int(worst["frame"]))

## Recap — API cheat sheet

| Task                                    | Call |
|-------------------------------------------|------|
| Locate a flight                            | `PathLoader(root).load_single_flight(flight_name=...)` / `StoutLoader().load_single_flight(...)` |
| Load drone + camera data                   | `flight.add_drone_data()`, `flight.add_camera_data(use_photogrammetry=False)` |
| Get the `Camera` instance                  | `flight.raw_data.payload_data.camera_obj` |
| Load pipeline config                       | `PhotogrammetryConfig(yaml_path, camera_model=...)` |
| Open/plot one frame                        | `open_frame(camera_obj, frame_number)` (Step 4.1) |
| Reload a finished result                   | `load_photogrammetry_result(flight, output_dir)` (Step 4.2) |
| Reload an intermediate checkpoint          | `load_checkpoint(flight, output_dir, step)` (Step 4.3) |
| Quick single-frame attitude + EKF check    | `quick_look(flight, camera_obj, cfg, output_dir, frame_number)` (Step 4.4) |
| Reload the latest saved EKF solution       | `load_ekf(flight)` (Step 4.4) |
| Run the pipeline from scratch              | `camera_obj.run_photogrammetry(csv_file, config, flight, output_dir=...)` (Step 5) |
| Resume from a checkpoint                   | same call + `start_from_dict=<path to dictionary_pN.ecsv>` (Step 6) |
| Run several flights from one campaign      | `Camera.run_photogrammetry_multi_flights(flights=[...], ...)` -- clicks targets once on the first flight, reuses them (via ORB alignment) for the rest |


# Debug Plots

In [ ]:
# # ── Isolated tracking test, seeded from a LIVE (verified-correct) init ─────
# # The earlier version of this test sourced its seed from dictionary_p2.ecsv,
# # which inherits the same frame-labeling issue we're investigating (its
# # first row is already one tracking step past the true init frame, not the
# # init frame itself). This version runs initialization fresh in-script and
# # captures the real frame_number _seek_to_frame returns -- no file, no
# # assumptions, no dependency on camera_obj's shared stream for the tracking
# # steps that follow.
# import cv2 as cv
# from IPA_flight.IPA_flight.finder import TargetFinder, process_frame
# from IPA_flight.IPA_flight.genParamFile import GenParamFile
# from IPA_flight.IPA_flight.drone import DroneData as IPADroneData

# N_FRAMES = 30

# params = GenParamFile.GeoPlot(
#     csv_file=str(TARGETS_CSV), reference_point=cfg.reference_point,
#     flight=flight, dji_base_logged=getattr(cfg, "dji_base_logged", None),
# )
# dh_wrapper = Camera._build_dh_wrapper(flight, camera_obj)
# target_finder = TargetFinder(
#     dh_wrapper, params["geodetic_targets"], params["geodetic_tel_positions"],
#     params["image_tel_positions"], cfg.camera_matrix, cfg.distortion_coeffs,
#     params["reference_point"], params["video_code"],
# )

# dd = IPADroneData(dh_wrapper, params["reference_point"], dirpath="/data/POLOCALC/campaigns/")
# _, _, req_start, req_end = dd.get_scanning_range(margin=1, plot=False, frame=True)
# start_frame, seed_frame = target_finder._seek_to_frame(int(req_start), int(req_end))
# print(f"requested start_frame={int(req_start)}, actual seek landed on frame {start_frame}")

# enu_targets = target_finder._geodetic_to_enu(target_finder.geodetic_targets)
# enu_tel     = target_finder._geodetic_to_enu(target_finder.geodetic_tel_positions)

# # This pops up the click UI -- click telescopes then targets, same as usual.
# targets_df, tel_pixels, tel_enu, rvec, tvec = target_finder.initialize_from_first_frame(
#     seed_frame, enu_targets, enu_tel, color_tol=cfg.finder.get("color_tol", 10)
# )
# targets_df, _ = target_finder.check_and_fix_initialization(seed_frame, targets_df)

# valid_mask     = targets_df[["x", "y", "R", "G", "B"]].notna().all(axis=1)
# valid_df       = targets_df[valid_mask].reset_index(drop=True)
# targets_pixels = valid_df[["x", "y", "R", "G", "B"]].to_numpy(dtype=np.float64)
# target_ids     = valid_df["target_id"].to_list()

# box_width  = cfg.finder.get("box_width", 10)
# box_height = cfg.finder.get("box_height", 10)
# tracking_method  = cfg.finder.get("tracking_method", "optical_flow")
# orb_nfeatures    = cfg.finder.get("orb_nfeatures", 10000)
# refine_threshold = cfg.finder.get("refine_threshold", 0.98)

# video_files = [p for p in camera_obj.path.iterdir() if p.suffix.lower() == ".mp4"]
# cap = cv.VideoCapture(str(video_files[0]))
# cap.set(cv.CAP_PROP_POS_FRAMES, start_frame)  # the REAL frame init happened on
# ok, check = cap.read()
# assert ok and np.array_equal(check, seed_frame), "capture didn't land on the true init frame!"

# frames_plotted = [(start_frame, seed_frame.copy(), targets_pixels[:, :2].copy())]
# prev_frame = seed_frame

# for step in range(1, N_FRAMES):
#     ok, frame = cap.read()
#     if not ok:
#         print(f"video exhausted at step {step}")
#         break
#     frame_number = start_frame + step  # correct by construction: our own sequential cap.read()

#     new_coords, _ = process_frame(
#         prev_frame, frame,
#         targets_pixels[:, :2], box_width, box_height, targets_pixels[:, 2:],
#         method=tracking_method, orb_nfeatures=orb_nfeatures,
#         refine_threshold=refine_threshold,
#     )
#     new_coords = np.array(new_coords, dtype=np.float64)
#     valid = ~np.isnan(new_coords).any(axis=1)
#     targets_pixels[valid, :2] = new_coords[valid]

#     frames_plotted.append((frame_number, frame.copy(), new_coords.copy()))
#     prev_frame = frame

# cap.release()

# # ── Plot grid ────────────────────────────────────────────────────────────
# cols = 6
# rows = int(np.ceil(len(frames_plotted) / cols))
# fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.6))
# axes = np.atleast_2d(axes)

# for ax, (frame_number, frame_img, coords) in zip(axes.flat, frames_plotted):
#     x, y = coords[:, 0], coords[:, 1]
#     missing = [tid for tid, xi in zip(target_ids, x) if np.isnan(xi)]

#     ax.imshow(cv.cvtColor(frame_img, cv.COLOR_BGR2RGB))
#     valid = ~np.isnan(x)
#     ax.scatter(x[valid], y[valid], s=30, facecolors="none", edgecolors="lime", linewidths=1.2)
#     ax.set_title(f"f{frame_number} — missing {len(missing)}"
#                  + (f"\n{missing}" if missing else ""),
#                  fontsize=7, color="red" if missing else "black")
#     ax.axis("off")

# for ax in axes.flat[len(frames_plotted):]:
#     ax.axis("off")

# plt.tight_layout()
# plt.show()
